# Youtube Data API: PlayListsItems
https://developers.google.com/youtube/v3/docs/playlistItems

## Contenido:

* [Librerias](*Librerias)
* [Videos](*Videos)


<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/', force_remount=True)

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## PlaylistItems

### Ejecucion

#### Parametros constantes

In [ ]:
URIpi='https://www.googleapis.com/youtube/v3/playlistItems?key='

# ================== Parametros Requeridos ==================

part = 'snippet,id,status,contentDetails' # Parametro obligatorio

# ================== Parametros opcionales ==================

maxResults = '50' # Numero maximo de elementos a devolver. El rango es 0 a 50.
# videoId = # Devolver elementos solo de la playlist que contienen este video

#### Parametros variables

In [ ]:
# ================== Filtros (especifique 0 o 1 de los siguientes parámetros) ==================

# id = # Lista separada por comas de ids de playlists
playlistId = 'PLU5ZtzFOQpMux1TCi_OLU4awDFBxG-z5a' # Id de la lista de reproduccion

# ================== Parametros opcionales ==================

pageToken = '' # Pagina especifica que se debe devolver

#### Solicitud

In [ ]:
r = requests.get(URIpi+key+'&part='+part+'&playlistId='+playlistId\
                 +'&maxResults='+maxResults+'&pageToken='+pageToken)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

In [ ]:
d['kind']

In [ ]:
d['etag']

In [ ]:
d['nextPageToken']

In [ ]:
d['pageInfo']

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en produccion

#### Importando Datos

In [ ]:
# ======================= OJO Cambiar PATH para cada Serie =======================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Search/\
Narcos/2022-09-13SV(2022-09-12)CR(2022-09-13)/PlayLists.csv' 

# ======================= OJO Cambiar PATH para cada Serie =======================

playlist=pd.read_csv(path)
playlist

In [ ]:
dataPL=playlist.loc[:,['idRecursoPL','Consulta','Pais']]
dataPL

#### Construimos lista de ids

In [ ]:
ListIds=list(dataPL['idRecursoPL'])
ListIds

#### Parametros constantes

In [ ]:
URIpi='https://www.googleapis.com/youtube/v3/playlistItems?key='

# ================== Parametros Requeridos ==================

part = 'snippet,id,status,contentDetails' # Parametro obligatorio
fields = 'pageInfo(*),nextPageToken(*),prevPageToken(*),\
items(contentDetails,snippet(title,description,playlistId,channelId,channelTitle),status)'

# ================== Parametros opcionales ==================

maxResults = '50' # Numero maximo de elementos a devolver. El rango es 0 a 50.

#### Parametros variables

In [ ]:
# ================== Filtros (especifique 0 o 1 de los siguientes parámetros) ==================

playlistId=ListIds # # Ids de las listas de reproduccion

# ================== Parametros opcionales ==================

pageToken='' # Pagina especifica que se debe devolver

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/PlayListsItems/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'PLAYLISTSITEMS'+str(datetime.now())

#### Constantes

In [ ]:
solicitud = 0 # Inicio de las solicitudes
paginasmax = 100 # Numero de paginas maximas
solicitudmax = 7000 # Solicitudes maximas, el limite es 10000
espera = 0.5 # Valor de retardo para las solicitudes

# ======================= OJO Cambiar diariamente =======================

ithId = 0 # i-esimo id

# ======================= OJO Cambiar diariamente =======================

#### Ejecucion

In [ ]:
Datac = pd.DataFrame() # Dataframe de Resultados Totales
Metadatac = pd.DataFrame(columns = ['Solicitud','IdRecursoPL','maxResultados','FechaEjecucion','Respuesta'\
                                    ,'prevPageToken','nextPageToken','ResultadosTotal','Resultadosxpagina']) # Dataframe de Informacion de Solicitudes

for id in playlistId[ithId:]: # For de lista de ids de playlist
  
  if (solicitud == solicitudmax+1):
    break
  
  pageToken = '' #pagina especifica que se debe devolver => OJO DEBE DE IR EN ESTE LUGAR

  for pagina in range(1,paginasmax+1): # For de la paginacion

    solicitud = solicitud+1

    if (solicitud == solicitudmax+1):
      ptoPartida = [] # Punto de partida para volver a comenzar el motor de busqueda en produccion.
      break

    time.sleep(espera) # Tiempo de espera por cada solicitud
    r = requests.get(URIpi+key+'&part='+part+'&fields='+fields+'&playlistId='+id\
                     +'&maxResults='+maxResults+'&pageToken='+pageToken) # Solicitud
    
    print('solicitud',solicitud,r.reason)

    if (r.reason == 'OK'): # Vemos si la solicitud fue exitosa

      d = dict(r.json())
      claves=list(d.keys())
      items = d.get('items')
      Data=pd.json_normalize(items).assign(Solicitud=solicitud)
      Datac=pd.concat([Datac, Data], ignore_index=True)

      if ('nextPageToken' in claves): # Si hay siguiente pagina
        pageToken = d['nextPageToken'] # Definimos el proximo pagetoken
        if ('prevPageToken' in claves):
          prevPageToken = d['prevPageToken']
        else:
          prevPageToken = ''

        Metadatac = Metadatac.append({'Solicitud':solicitud,'IdRecursoPL':id,'maxResultados':maxResults,'FechaEjecucion':str(datetime.now())\
                                      ,'Respuesta':r.reason,'nextPageToken':d['nextPageToken'],'prevPageToken':prevPageToken\
                                      ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index = True)
        print("Siguiente pagina")
        
      else: # Si no hay siguiente pagina
        nextPageToken = '' # Definimos el nextPageToken como vacio

        if ('prevPageToken' in claves):
          prevPageToken = d['prevPageToken']
        else:
          prevPageToken = ''

        Metadatac = Metadatac.append({'Solicitud':solicitud,'IdRecursoPL':id,'maxResultados':maxResults,'FechaEjecucion':str(datetime.now())\
                                      ,'Respuesta':r.reason,'nextPageToken':nextPageToken,'prevPageToken':prevPageToken\
                                      ,'ResultadosTotal':d['pageInfo']['totalResults'],'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index = True)
        print("No hay mas paginas")
        break
    else: # Vemos si la solicitud no fue exitosa    
      print("No fue exitosa la solicitud", solicitud)  
print("Busqueda Terminada")

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

#### Estructurar y dar formato a Dataframe

In [ ]:
columnas={'contentDetails.videoId':'idRecursoVD','snippet.title':'Titulo','snippet.description':'Descripcion'\
          ,'contentDetails.videoPublishedAt':'TiempoPublicacion','status.privacyStatus':'TipoPrivacidad','snippet.channelId':'idCanal'\
          ,'snippet.channelTitle':'TituloCanal','Solicitud':'Solicitud','snippet.playlistId':'idRecursoPL'}

columnasorden=['idRecursoVD','Titulo','Descripcion','TiempoPublicacion','TipoPrivacidad'\
               ,'idRecursoPL','idCanal','TituloCanal','Solicitud']

Datac.rename(columns=columnas,inplace=True)
DATA=Datac.reindex(columns=columnasorden)
DATA

#### Uniendo columnas de importancia

In [ ]:
mergedinner = pd.merge(left=DATA,right=dataPL, left_on='idRecursoPL', right_on='idRecursoPL')
mergedinner.info()

In [ ]:
mergedinner

In [ ]:
columnasorden=['idRecursoVD','Titulo','Descripcion','TiempoPublicacion','TipoPrivacidad'\
               ,'idRecursoPL','idCanal','TituloCanal','Consulta','Pais','Solicitud']
DATA=mergedinner.reindex(columns=columnasorden)
DATA

#### Extraccion de datos

Se generan 4 nuevas columnas (emoticones, hashtags, URLs y menciones) por cada columna de texto importante. (En este caso 8)

##### Extreaer emoticones

In [ ]:
def emojis(texto):
  if (len(texto) != 0):
    emoji_summary = adv.extract_emoji(texto)
    return emoji_summary['top_emoji']
  else:
    L=[]
    return L

In [ ]:
DATA['emojisTitulo']=DATA['Titulo'].apply(emojis)

In [ ]:
DATA['emojisTitulo']

In [ ]:
DATA['emojisDescripcion']=DATA['Descripcion'].apply(emojis)

In [ ]:
DATA['emojisDescripcion']

##### Extraer URLs

In [ ]:
extractor = URLExtract()
def URLs(texto):
  URL=extractor.find_urls(texto)
  return URL

In [ ]:
DATA['URLsDescripcion']=DATA['Descripcion'].apply(URLs)
DATA['URLsTitulo']=DATA['Titulo'].apply(URLs)

In [ ]:
DATA['URLsTitulo']

In [ ]:
DATA['URLsDescripcion']

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['Titulo'].apply(codigohex)
DATA['DescripcionModificado']=DATA['Descripcion'].apply(codigohex)

##### Extraer Hastags

In [ ]:
def hastags(texto):
  pattern='#\w+'
  hastag=regexp_tokenize(texto,pattern)
  return hastag

In [ ]:
DATA['hastagsDescripcion']=DATA['DescripcionModificado'].apply(hastags)
DATA['hastagsTitulo']=DATA['TituloModificado'].apply(hastags)

In [ ]:
DATA['hastagsTitulo']

In [ ]:
DATA['hastagsDescripcion']

##### Extraer menciones o etiquetas (puede ser necesario alguna modificacion para el caso de correos)

In [ ]:
def mencion(texto):
  pattern='@\w+'
  mencion=regexp_tokenize(texto,pattern)
  return mencion

In [ ]:
DATA['MencionesDescripcion']=DATA['Descripcion'].apply(mencion)
DATA['MencionesTitulo']=DATA['Titulo'].apply(mencion)

In [ ]:
DATA['MencionesTitulo']

In [ ]:
DATA['MencionesDescripcion']

In [ ]:
DATA.info()

#### Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos)

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['Titulo'].apply(codigohex)
DATA['DescripcionModificado']=DATA['Descripcion'].apply(codigohex)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropunicode)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropunicode)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar secuencias de escape

In [ ]:
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\r', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\n', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\v', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\t', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\f', ' '))
DATA['Titulo'] = DATA['Titulo'].apply(str.replace, args=('\b', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\r', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\n', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\v', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\t', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\f', ' '))
DATA['TituloModificado'] = DATA['TituloModificado'].apply(str.replace, args=('\b', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\r', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\n', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\v', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\t', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\f', ' '))
DATA['Descripcion'] = DATA['Descripcion'].apply(str.replace, args=('\b', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\r', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\n', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\v', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\t', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\f', ' '))
DATA['DescripcionModificado'] = DATA['DescripcionModificado'].apply(str.replace, args=('\b', ' '))

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropURLs)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropURLs)
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropURL)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropURL)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropemoticons)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropemoticons)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropemails)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropemails)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropmenciones)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropmenciones)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(droppalabrasrepetidas)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(droppalabrasrepetidas)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropnumeros)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropnumeros)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar \# de Hashtags

In [ ]:
def dropnumeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropnumeral)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropnumeral)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Dividir palabras adjuntas (CamelCase)

In [ ]:
def splitpalabras(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(splitpalabras)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(splitpalabras)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation) 
punct 

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(puntuacion)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(puntuacion)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(dropespacios)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(dropespacios)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(droppalabrascortas)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(droppalabrascortas)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
DATA['TituloModificado']=DATA['TituloModificado'].apply(normatext)
DATA['DescripcionModificado']=DATA['DescripcionModificado'].apply(normatext)

In [ ]:
DATA['TituloModificado']

In [ ]:
DATA['DescripcionModificado']

#### Estructurar y dar formato

In [ ]:
DATA.info()

In [ ]:
columnasorden=['idRecursoVD','Titulo','TituloModificado', 'emojisTitulo', 'URLsTitulo', 'hastagsTitulo','MencionesTitulo'\
               ,'Descripcion','DescripcionModificado', 'emojisDescripcion', 'URLsDescripcion', 'hastagsDescripcion','MencionesDescripcion','TiempoPublicacion','TipoPrivacidad'\
               ,'idRecursoPL','idCanal','TituloCanal','Consulta','Pais','Solicitud']

DATA=DATA.reindex(columns=columnasorden)
DATA

In [ ]:
DATA.info()

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)